# Cumplimiento Regulatorio CNSF

## RCS (Capital de Solvencia) y Reservas Técnicas

La Comisión Nacional de Seguros y Fianzas (CNSF) establece requisitos de capital mínimo y reservas técnicas. Este notebook demuestra el cálculo del Requerimiento de Capital de Solvencia (RCS) para productos de Vida y Daños.

### Contenido
1. Introducción al RCS y marco regulatorio
2. RCS para Seguros de Vida
3. RCS para Seguros de Daños
4. Reservas Técnicas (Matemática y Riesgos en Curso)
5. Validación de Suficiencia de Capital
6. Dashboard de Solvencia

In [ ]:
# Imports necesarios
from decimal import Decimal
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Imports regulatorios
from mexican_insurance.regulatorio import (
    RCSVida,
    RCSDanos,
    ReservaMatematica,
    ReservaRiesgosCurso,
    ValidadorSuficiencia
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Imports completados exitosamente")

## 1. Introducción al RCS y Marco Regulatorio

### Requerimiento de Capital de Solvencia (RCS)

El RCS es el capital mínimo que debe mantener una aseguradora para absorber pérdidas inesperadas. Se calcula considerando:

- **Riesgo de Suscripción**: Variabilidad en siniestros y primas
- **Riesgo de Mercado**: Inversiones, tasas de interés, tipo de cambio
- **Riesgo de Crédito**: Contrapartes, reaseguradores
- **Riesgo Operacional**: Fallas en procesos, sistemas, fraude

### Suficiencia de Capital

$$\text{Ratio de Solvencia} = \frac{\text{Capital Disponible}}{\text{RCS}} \geq 100\%$$

- **< 100%**: Insuficiente - Plan de regularización
- **100-150%**: Adecuado - Vigilancia
- **> 150%**: Excedente - Solvente

## 2. RCS para Seguros de Vida

El RCS de Vida considera mortalidad, longevidad, rescates, gastos y riesgo de tasa de interés.

In [ ]:
# Datos de ejemplo para cartera de vida
datos_vida = {
    'suma_asegurada_total': Decimal('500000000'),  # 500M en suma asegurada
    'reserva_matematica': Decimal('150000000'),    # 150M en reservas
    'primas_anuales': Decimal('50000000'),         # 50M en primas anuales
    'num_polizas': 5000,
    'duracion_media': 12.5,  # años
    'tasa_tecnica': Decimal('0.055'),  # 5.5%
}

print("📋 CARTERA DE SEGUROS DE VIDA")
print("="*70)
print(f"Suma Asegurada Total:   ${datos_vida['suma_asegurada_total']:>15,.0f}")
print(f"Reserva Matemática:     ${datos_vida['reserva_matematica']:>15,.0f}")
print(f"Primas Anuales:         ${datos_vida['primas_anuales']:>15,.0f}")
print(f"Número de Pólizas:      {datos_vida['num_polizas']:>15,}")
print(f"Duración Media:         {datos_vida['duracion_media']:>15.1f} años")

In [ ]:
# Calcular RCS de Vida
rcs_vida = RCSVida(
    suma_asegurada=datos_vida['suma_asegurada_total'],
    reserva_matematica=datos_vida['reserva_matematica'],
    primas_anuales=datos_vida['primas_anuales'],
    duracion_media=datos_vida['duracion_media']
)

resultado_rcs_vida = rcs_vida.calcular()

print("💰 REQUERIMIENTO DE CAPITAL DE SOLVENCIA - VIDA")
print("="*70)
print(f"\nComponentes de Riesgo:")
print(f"  Riesgo de Mortalidad:    ${resultado_rcs_vida['riesgo_mortalidad']:>15,.0f}")
print(f"  Riesgo de Longevidad:    ${resultado_rcs_vida['riesgo_longevidad']:>15,.0f}")
print(f"  Riesgo de Rescates:      ${resultado_rcs_vida['riesgo_rescates']:>15,.0f}")
print(f"  Riesgo de Gastos:        ${resultado_rcs_vida['riesgo_gastos']:>15,.0f}")
print(f"  Riesgo de Tasa:          ${resultado_rcs_vida['riesgo_tasa']:>15,.0f}")

print(f"\n\nRCS Agregado (con correlaciones):")
print(f"  RCS Total:               ${resultado_rcs_vida['rcs_total']:>15,.0f}")

# Cálculo de ratio sobre reserva
ratio_reserva = (resultado_rcs_vida['rcs_total'] / datos_vida['reserva_matematica'] * 100)
print(f"\n  RCS / Reserva:           {ratio_reserva:>15.2f}%")

In [ ]:
# Visualización del RCS Vida
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfica 1: Composición del RCS
componentes = ['Mortalidad', 'Longevidad', 'Rescates', 'Gastos', 'Tasa']
valores = [
    float(resultado_rcs_vida['riesgo_mortalidad']),
    float(resultado_rcs_vida['riesgo_longevidad']),
    float(resultado_rcs_vida['riesgo_rescates']),
    float(resultado_rcs_vida['riesgo_gastos']),
    float(resultado_rcs_vida['riesgo_tasa'])
]

colores = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
wedges, texts, autotexts = ax1.pie(valores, labels=componentes, autopct='%1.1f%%',
                                     startangle=90, colors=colores, textprops={'fontsize': 11})
ax1.set_title('Composición del RCS - Vida\nPor Tipo de Riesgo', 
              fontsize=13, fontweight='bold')

# Gráfica 2: Comparación suma simple vs agregado
suma_simple = sum(valores)
rcs_agregado = float(resultado_rcs_vida['rcs_total'])
beneficio_diversificacion = suma_simple - rcs_agregado

categorias = ['Suma Simple', 'RCS Agregado\n(con correlaciones)']
valores_comp = [suma_simple, rcs_agregado]
colors_comp = ['#e74c3c', '#2ecc71']

bars = ax2.bar(categorias, valores_comp, color=colors_comp, alpha=0.7, 
               edgecolor='black', linewidth=2)
ax2.set_ylabel('Capital Requerido (MXN)', fontsize=12)
ax2.set_title('Beneficio de Diversificación\nRCS Vida', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.0f}M'))

# Agregar valores
for bar, val in zip(bars, valores_comp):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'${val/1e6:.1f}M',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Mostrar beneficio
ax2.text(0.5, suma_simple * 0.5, 
         f'Beneficio\nDiversificación\n${beneficio_diversificacion/1e6:.1f}M',
         ha='center', va='center', fontsize=11, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

plt.tight_layout()
plt.show()

## 3. RCS para Seguros de Daños

El RCS de Daños considera prima, siniestralidad, volatilidad y riesgo catastrófico.

In [ ]:
# Datos de ejemplo para cartera de daños
datos_danos = {
    'prima_emitida_anual': Decimal('80000000'),    # 80M en primas
    'siniestros_ultimo_año': Decimal('55000000'),  # 55M en siniestros
    'reserva_siniestros': Decimal('40000000'),     # 40M en reservas
    'num_polizas': 12000,
    'loss_ratio_historico': Decimal('0.70'),       # 70%
    'volatilidad': Decimal('0.15'),                # 15%
    'exposicion_catastrofe': Decimal('200000000'), # 200M PML
}

print("📋 CARTERA DE SEGUROS DE DAÑOS")
print("="*70)
print(f"Prima Emitida Anual:        ${datos_danos['prima_emitida_anual']:>15,.0f}")
print(f"Siniestros Último Año:      ${datos_danos['siniestros_ultimo_año']:>15,.0f}")
print(f"Reserva de Siniestros:      ${datos_danos['reserva_siniestros']:>15,.0f}")
print(f"Número de Pólizas:          {datos_danos['num_polizas']:>15,}")
print(f"Loss Ratio Histórico:       {datos_danos['loss_ratio_historico']*100:>15.1f}%")
print(f"Exposición Catastrófica:    ${datos_danos['exposicion_catastrofe']:>15,.0f}")

In [ ]:
# Calcular RCS de Daños
rcs_danos = RCSDanos(
    prima_emitida=datos_danos['prima_emitida_anual'],
    siniestros=datos_danos['siniestros_ultimo_año'],
    reserva_siniestros=datos_danos['reserva_siniestros'],
    volatilidad=datos_danos['volatilidad'],
    exposicion_catastrofe=datos_danos['exposicion_catastrofe']
)

resultado_rcs_danos = rcs_danos.calcular()

print("💰 REQUERIMIENTO DE CAPITAL DE SOLVENCIA - DAÑOS")
print("="*70)
print(f"\nComponentes de Riesgo:")
print(f"  Riesgo de Prima:         ${resultado_rcs_danos['riesgo_prima']:>15,.0f}")
print(f"  Riesgo de Reserva:       ${resultado_rcs_danos['riesgo_reserva']:>15,.0f}")
print(f"  Riesgo Catastrófico:     ${resultado_rcs_danos['riesgo_catastrofe']:>15,.0f}")

print(f"\n\nRCS Agregado (con correlaciones):")
print(f"  RCS Total:               ${resultado_rcs_danos['rcs_total']:>15,.0f}")

# Cálculo de ratio sobre prima
ratio_prima = (resultado_rcs_danos['rcs_total'] / datos_danos['prima_emitida_anual'] * 100)
print(f"\n  RCS / Prima Emitida:     {ratio_prima:>15.2f}%")

In [ ]:
# Visualización del RCS Daños
fig, ax = plt.subplots(figsize=(12, 6))

# Gráfica de barras por componente
componentes_danos = ['Riesgo de\nPrima', 'Riesgo de\nReserva', 'Riesgo\nCatastrófico']
valores_danos = [
    float(resultado_rcs_danos['riesgo_prima']),
    float(resultado_rcs_danos['riesgo_reserva']),
    float(resultado_rcs_danos['riesgo_catastrofe'])
]

colores_danos = ['#3498db', '#e74c3c', '#f39c12']
bars = ax.bar(componentes_danos, valores_danos, color=colores_danos, 
              alpha=0.7, edgecolor='black', linewidth=2)

# Línea horizontal del RCS total
ax.axhline(y=float(resultado_rcs_danos['rcs_total']), color='green', 
           linestyle='--', linewidth=2.5, 
           label=f"RCS Total: ${float(resultado_rcs_danos['rcs_total'])/1e6:.1f}M")

ax.set_ylabel('Capital Requerido (MXN)', fontsize=12)
ax.set_title('Composición del RCS - Daños\nPor Tipo de Riesgo', 
             fontsize=13, fontweight='bold', pad=20)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.0f}M'))

# Agregar valores sobre las barras
for bar, val in zip(bars, valores_danos):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'${val/1e6:.1f}M',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Reservas Técnicas

Calculemos las reservas técnicas según la normativa CNSF.

In [ ]:
# A. Reserva Matemática (Vida)
print("📊 RESERVA MATEMÁTICA - VIDA")
print("="*70)

# Cargar cartera de vida
cartera_vida_df = pd.read_csv('../examples/data/cartera_ejemplo.csv')
cartera_vida_df = cartera_vida_df[cartera_vida_df['producto'].str.contains('Vida')]

print(f"\nPólizas de Vida: {len(cartera_vida_df)}")
print(f"Suma Asegurada Total: ${cartera_vida_df['suma_asegurada'].sum():,.0f}")
print(f"Prima Mensual Total: ${cartera_vida_df['prima_mensual'].sum():,.0f}")

# Calcular reserva matemática por póliza
reserva_mat = ReservaMatematica()
cartera_vida_df['reserva_matematica'] = cartera_vida_df.apply(
    lambda row: reserva_mat.calcular(
        suma_asegurada=Decimal(str(row['suma_asegurada'])),
        edad_actual=row['edad'],
        plazo_restante=row['plazo_años'] - 1,  # Asumimos 1 año transcurrido
        tasa_tecnica=Decimal('0.055')
    ),
    axis=1
)

total_reserva_matematica = cartera_vida_df['reserva_matematica'].sum()
print(f"\n💰 RESERVA MATEMÁTICA TOTAL: ${total_reserva_matematica:,.0f}")
print(f"   Promedio por póliza: ${total_reserva_matematica/len(cartera_vida_df):,.0f}")

In [ ]:
# B. Reserva de Riesgos en Curso (Daños)
print("📊 RESERVA DE RIESGOS EN CURSO - DAÑOS")
print("="*70)

# Simular pólizas de daños
polizas_danos = {
    'Autos': {'prima_neta': Decimal('30000000'), 'dias_transcurridos': 120, 'dias_vigencia': 365},
    'Hogar': {'prima_neta': Decimal('15000000'), 'dias_transcurridos': 90, 'dias_vigencia': 365},
    'GMM': {'prima_neta': Decimal('25000000'), 'dias_transcurridos': 150, 'dias_vigencia': 365},
    'RC': {'prima_neta': Decimal('10000000'), 'dias_transcurridos': 180, 'dias_vigencia': 365},
}

reserva_rc = ReservaRiesgosCurso()
total_reserva_rc = Decimal('0')

print("\nReserva por Ramo:")
for ramo, datos in polizas_danos.items():
    reserva = reserva_rc.calcular(
        prima_neta=datos['prima_neta'],
        dias_transcurridos=datos['dias_transcurridos'],
        dias_vigencia=datos['dias_vigencia']
    )
    total_reserva_rc += reserva
    pct_no_devengado = (1 - datos['dias_transcurridos'] / datos['dias_vigencia']) * 100
    print(f"  {ramo:10s}: ${reserva:>12,.0f}  ({pct_no_devengado:.1f}% no devengado)")

print(f"\n{'='*70}")
print(f"💰 RESERVA DE RIESGOS EN CURSO TOTAL: ${total_reserva_rc:,.0f}")
print(f"{'='*70}")

In [ ]:
# Visualización de reservas técnicas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfica 1: Reserva Matemática por producto
reserva_por_producto = cartera_vida_df.groupby('producto')['reserva_matematica'].sum()
ax1.barh(reserva_por_producto.index, reserva_por_producto.values, 
         color='#3498db', alpha=0.7, edgecolor='black')
ax1.set_xlabel('Reserva Matemática (MXN)', fontsize=12)
ax1.set_title('Reserva Matemática por Producto\nCartera de Vida', 
              fontsize=13, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

# Agregar valores
for i, (prod, val) in enumerate(reserva_por_producto.items()):
    ax1.text(val, i, f'  ${val/1e6:.2f}M', va='center', fontsize=10, fontweight='bold')

# Gráfica 2: Reserva de Riesgos en Curso
ramos = list(polizas_danos.keys())
reservas_rc_values = []
for ramo, datos in polizas_danos.items():
    res = reserva_rc.calcular(
        prima_neta=datos['prima_neta'],
        dias_transcurridos=datos['dias_transcurridos'],
        dias_vigencia=datos['dias_vigencia']
    )
    reservas_rc_values.append(float(res))

bars = ax2.bar(ramos, reservas_rc_values, color='#e74c3c', alpha=0.7, 
               edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Reserva de Riesgos en Curso (MXN)', fontsize=12)
ax2.set_title('Reserva de Riesgos en Curso por Ramo\nCartera de Daños', 
              fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.0f}M'))

# Agregar valores
for bar, val in zip(bars, reservas_rc_values):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'${val/1e6:.1f}M',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Validación de Suficiencia de Capital

Validemos que la aseguradora cumple con los requisitos de solvencia.

In [ ]:
# Datos de la aseguradora
capital_aseguradora = {
    'capital_pagado': Decimal('100000000'),        # 100M
    'reservas_capital': Decimal('50000000'),       # 50M
    'utilidades_retenidas': Decimal('30000000'),   # 30M
    'deuda_subordinada': Decimal('20000000'),      # 20M (máx 50% del básico)
}

# Capital disponible
capital_basico = (capital_aseguradora['capital_pagado'] + 
                  capital_aseguradora['reservas_capital'] + 
                  capital_aseguradora['utilidades_retenidas'])
capital_complementario = capital_aseguradora['deuda_subordinada']
capital_total = capital_basico + capital_complementario

# RCS Total (sumando Vida + Daños con correlación)
# Asumimos correlación 0.25 entre Vida y Daños
rcs_vida_val = float(resultado_rcs_vida['rcs_total'])
rcs_danos_val = float(resultado_rcs_danos['rcs_total'])
correlacion = 0.25
rcs_total_combinado = np.sqrt(
    rcs_vida_val**2 + rcs_danos_val**2 + 
    2 * correlacion * rcs_vida_val * rcs_danos_val
)

print("📊 VALIDACIÓN DE SUFICIENCIA DE CAPITAL")
print("="*70)
print(f"\nCapital Disponible:")
print(f"  Capital Pagado:          ${capital_aseguradora['capital_pagado']:>15,.0f}")
print(f"  Reservas de Capital:     ${capital_aseguradora['reservas_capital']:>15,.0f}")
print(f"  Utilidades Retenidas:    ${capital_aseguradora['utilidades_retenidas']:>15,.0f}")
print(f"  {'─'*50}")
print(f"  Capital Básico:          ${capital_basico:>15,.0f}")
print(f"  Deuda Subordinada:       ${capital_complementario:>15,.0f}")
print(f"  {'─'*50}")
print(f"  CAPITAL TOTAL:           ${capital_total:>15,.0f}")

print(f"\n\nRequerimiento de Capital:")
print(f"  RCS Vida:                ${rcs_vida_val:>15,.0f}")
print(f"  RCS Daños:               ${rcs_danos_val:>15,.0f}")
print(f"  {'─'*50}")
print(f"  RCS TOTAL (agregado):    ${rcs_total_combinado:>15,.0f}")

# Calcular ratio de solvencia
ratio_solvencia = (float(capital_total) / rcs_total_combinado * 100)
excedente = float(capital_total) - rcs_total_combinado

print(f"\n\n{'='*70}")
print(f"RATIO DE SOLVENCIA: {ratio_solvencia:.2f}%")
print(f"EXCEDENTE DE CAPITAL: ${excedente:,.0f}")
print(f"{'='*70}")

# Validación
validador = ValidadorSuficiencia()
resultado_validacion = validador.validar(
    capital_disponible=capital_total,
    rcs_requerido=Decimal(str(rcs_total_combinado))
)

print(f"\n📋 RESULTADO DE VALIDACIÓN:")
print(f"  Estado: {resultado_validacion['estado']}")
print(f"  Cumple: {'✅ SÍ' if resultado_validacion['cumple'] else '❌ NO'}")
print(f"  Observaciones: {resultado_validacion['observaciones']}")

## 6. Dashboard de Solvencia

Creemos un dashboard visual para monitorear la solvencia.

In [ ]:
# Dashboard completo de solvencia
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Gauge de Ratio de Solvencia
ax1 = fig.add_subplot(gs[0, :])
ax1.axis('off')

# Crear gauge manual
theta = np.linspace(0, np.pi, 100)
r = 1
x = r * np.cos(theta)
y = r * np.sin(theta)

# Zonas de color
ax1_gauge = fig.add_subplot(gs[0, :], projection='polar')
theta_ranges = [0, np.pi/3, 2*np.pi/3, np.pi]
colors_gauge = ['red', 'yellow', 'green']
labels_gauge = ['< 100%', '100-150%', '> 150%']

for i in range(len(theta_ranges)-1):
    theta_section = np.linspace(theta_ranges[i], theta_ranges[i+1], 50)
    ax1_gauge.fill_between(theta_section, 0, 1, color=colors_gauge[i], alpha=0.3)

# Aguja indicadora
ratio_normalized = min(ratio_solvencia / 200, 1.0)  # Normalizar a rango 0-200%
angle = np.pi * (1 - ratio_normalized)
ax1_gauge.plot([angle, angle], [0, 0.9], 'k-', linewidth=4)
ax1_gauge.plot(angle, 0.9, 'ko', markersize=15)

ax1_gauge.set_ylim(0, 1)
ax1_gauge.set_theta_zero_location('W')
ax1_gauge.set_theta_direction(1)
ax1_gauge.set_xticks([0, np.pi/3, 2*np.pi/3, np.pi])
ax1_gauge.set_xticklabels(['200%', '150%', '100%', '0%'])
ax1_gauge.set_yticks([])
ax1_gauge.set_title(f'RATIO DE SOLVENCIA: {ratio_solvencia:.1f}%', 
                    fontsize=16, fontweight='bold', pad=20)

# 2. Capital vs RCS
ax2 = fig.add_subplot(gs[1, 0])
categorias_cap = ['Capital\nDisponible', 'RCS\nRequerido']
valores_cap = [float(capital_total), rcs_total_combinado]
colors_cap = ['#2ecc71', '#e74c3c']

bars = ax2.bar(categorias_cap, valores_cap, color=colors_cap, alpha=0.7, 
               edgecolor='black', linewidth=2)
ax2.set_ylabel('Millones MXN', fontsize=11)
ax2.set_title('Capital vs RCS', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.0f}M'))

for bar, val in zip(bars, valores_cap):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'${val/1e6:.0f}M', ha='center', va='bottom', 
            fontsize=10, fontweight='bold')

# 3. Composición del Capital
ax3 = fig.add_subplot(gs[1, 1])
capital_items = ['Capital\nPagado', 'Reservas', 'Utilidades', 'Deuda\nSubordinada']
capital_values = [
    float(capital_aseguradora['capital_pagado']),
    float(capital_aseguradora['reservas_capital']),
    float(capital_aseguradora['utilidades_retenidas']),
    float(capital_aseguradora['deuda_subordinada'])
]

wedges, texts, autotexts = ax3.pie(capital_values, labels=capital_items, 
                                     autopct='%1.1f%%', startangle=90,
                                     colors=['#3498db', '#2ecc71', '#f39c12', '#9b59b6'],
                                     textprops={'fontsize': 9})
ax3.set_title('Composición del Capital', fontsize=12, fontweight='bold')

# 4. RCS por Línea de Negocio
ax4 = fig.add_subplot(gs[1, 2])
lineas = ['Vida', 'Daños']
rcs_lineas = [rcs_vida_val, rcs_danos_val]
colors_lineas = ['#3498db', '#e74c3c']

bars = ax4.barh(lineas, rcs_lineas, color=colors_lineas, alpha=0.7, 
                edgecolor='black', linewidth=2)
ax4.set_xlabel('RCS (Millones MXN)', fontsize=11)
ax4.set_title('RCS por Línea de Negocio', fontsize=12, fontweight='bold')
ax4.grid(axis='x', alpha=0.3)
ax4.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.0f}M'))

for i, (bar, val) in enumerate(zip(bars, rcs_lineas)):
    width = bar.get_width()
    ax4.text(width, i, f'  ${val/1e6:.0f}M', va='center', 
             fontsize=10, fontweight='bold')

# 5. Reservas Técnicas
ax5 = fig.add_subplot(gs[2, 0])
reservas_items = ['Reserva\nMatemática', 'Reserva\nRiesgos en Curso']
reservas_values = [float(total_reserva_matematica), float(total_reserva_rc)]
colors_res = ['#3498db', '#e74c3c']

bars = ax5.bar(reservas_items, reservas_values, color=colors_res, 
               alpha=0.7, edgecolor='black', linewidth=2)
ax5.set_ylabel('Millones MXN', fontsize=11)
ax5.set_title('Reservas Técnicas', fontsize=12, fontweight='bold')
ax5.grid(axis='y', alpha=0.3)
ax5.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.0f}M'))

for bar, val in zip(bars, reservas_values):
    height = bar.get_height()
    ax5.text(bar.get_x() + bar.get_width()/2., height,
            f'${val/1e6:.0f}M', ha='center', va='bottom', 
            fontsize=10, fontweight='bold')

# 6. Resumen de cumplimiento
ax6 = fig.add_subplot(gs[2, 1:])
ax6.axis('off')

# Crear tabla de resumen
resumen_data = [
    ['Métrica', 'Valor', 'Estado'],
    ['Ratio de Solvencia', f'{ratio_solvencia:.1f}%', '✅ Solvente'],
    ['Capital Disponible', f'${float(capital_total)/1e6:.1f}M', ''],
    ['RCS Requerido', f'${rcs_total_combinado/1e6:.1f}M', ''],
    ['Excedente', f'${excedente/1e6:.1f}M', ''],
    ['Reserva Matemática', f'${float(total_reserva_matematica)/1e6:.1f}M', '✅ Constituida'],
    ['Reserva Riesgos en Curso', f'${float(total_reserva_rc)/1e6:.1f}M', '✅ Constituida'],
]

table = ax6.table(cellText=resumen_data, cellLoc='left', loc='center',
                  colWidths=[0.4, 0.3, 0.3])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

# Estilizar header
for i in range(3):
    table[(0, i)].set_facecolor('#3498db')
    table[(0, i)].set_text_props(weight='bold', color='white')

ax6.set_title('Resumen de Cumplimiento Regulatorio', 
              fontsize=12, fontweight='bold', pad=20)

plt.suptitle('Dashboard de Solvencia - CNSF', fontsize=18, fontweight='bold', y=0.98)
plt.show()

print("\n✅ Dashboard de solvencia generado exitosamente")

## Conclusiones

En este notebook aprendimos:

1. **RCS de Vida**: Componentes de mortalidad, longevidad, rescates, gastos y tasa
   - Importancia de la duración de la cartera
   - Beneficios de diversificación

2. **RCS de Daños**: Prima, reserva y riesgo catastrófico
   - Volatilidad de la cartera
   - Exposición a eventos extremos

3. **Reservas Técnicas**:
   - Reserva Matemática para productos de vida
   - Reserva de Riesgos en Curso para productos de daños

4. **Validación de Suficiencia**:
   - Ratio de solvencia mínimo 100%
   - Composición del capital (básico + complementario)
   - Monitoreo continuo

### Requisitos Regulatorios Clave:
- Ratio de Solvencia ≥ 100%
- Reportes trimestrales a CNSF
- Reservas técnicas suficientes
- Plan de capitalización si ratio < 100%

### Próximos Pasos

Continúa con:
- **05_validaciones_sat.ipynb**: Validaciones fiscales
- **06_reportes_cnsf.ipynb**: Generación automatizada de reportes
- **07_casos_practicos_completos.ipynb**: Workflows end-to-end